# 05 — Full-corpus evaluation and promotion gate

Evaluate the unified checkpoint on the complete independent validation and test splits. Every available data class participates in the same metrics; training remains isolated to the train split. Both evaluation tracks preserve the same checkpoint and tokenizer hashes.

In [ ]:
import sys, json
from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "phone-research.yaml").is_file() and (candidate / "src" / "onecore").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)
sys.path.insert(0, str(root / "src"))

from onecore.checkpoint import write_json_atomic
from onecore.evaluate import evaluate_checkpoint, promotion_gate
from onecore.promote import promote_candidate

config = root / "configs" / "phone-research.yaml"
checkpoint = root / "run-state" / "checkpoint.pt"
tokenizer = root / "run-state" / "tokenizer.json"
thresholds = root / "configs" / "promotion-thresholds.json"

validation_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.validation.json", split="validation",
)
test_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.test.json", split="test",
)

metrics = dict(validation_metrics)
metrics["test_metrics"] = test_metrics
metrics["test_manifest_sha256"] = test_metrics.get("split_manifest_sha256")
metrics["test_split_independent"] = (
    validation_metrics.get("split_manifest_sha256") != test_metrics.get("split_manifest_sha256")
)
metrics["evaluation_bundle_id"] = (
    f"{validation_metrics['evaluation_id']}+{test_metrics['evaluation_id']}"
)
write_json_atomic(root / "run-state" / "evaluation.json", metrics)

baseline_path = root / "run-state" / "evaluation.baseline.json"
baseline_metrics = (
    json.loads(baseline_path.read_text(encoding="utf-8"))
    if baseline_path.is_file() else None
)
passed, failures = promotion_gate(metrics, baseline=baseline_metrics)
print(json.dumps({
    "external_validation": validation_metrics,
    "external_test": test_metrics,
    "baseline_evaluation_loaded": baseline_metrics is not None,
    "baseline_evaluation_path": str(baseline_path) if baseline_metrics is not None else None,
    "promotion_gate_passed": passed,
    "failures": failures,
}, ensure_ascii=False, indent=2))
if passed:
    print(promote_candidate(root / "run-state" / "evaluation.json", root / "run-state" / "model-registry.json"))
else:
    print("PROMOTION BLOCKED. No candidate was merged or marked as released.")
